### From user inputs to Cypher queries

<img src="image1.png">

#### Cypher translator has 2 roles:
<pre>
C'est le langage de requête créé par Neo4j pour interagir avec les bases de données orientées graphe

    - 1. convert user query to a cypher query 

    - 2. extract data from graph database (Neo4j) then the llm convert the responses to the natural language for the user

In [1]:
# Code précédent:

# chargement du fichier.pdf

from langchain_community.document_loaders import PyPDFLoader

loader =  PyPDFLoader("pdf_file_example.pdf")
documents = loader.load()





from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm = ChatOllama(model= "llama3.2", temperature= 0)
llm_transformer = LLMGraphTransformer(llm = llm)

graph_documents = llm_transformer.convert_to_graph_documents(documents)




# Connexion à la base de donnée Neoj4

from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url="neo4j+s://5f0bd404.databases.neo4j.io",
    username="5f0bd404",
    password="Tz9KM4a8ddrglvbVPZ4E2m2VXqQL0FKinQiijyWBQh0",
    database="5f0bd404"
)





# storing graph documents


# Initialisation du llm local via ollama
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm = ChatOllama(model= "llama3.2", temperature= 0)


# convertir le texte vers graphe
llm_transformer = LLMGraphTransformer(llm = llm)



# extraction des noeuds et relations depuis le document
graph_documents = llm_transformer.convert_to_graph_documents(documents)


# Injection dans la base Neoj4
graph.add_graph_documents(
    graph_documents,
include_source=True,     # Lie le nœud Document d'origine aux entités créées
    baseEntityLabel=True      # Ajoute une étiquette générique `__Entity__` sur chaque nœud
)


print("Graphe importé avec succés: ")





C:\Users\hp\AppData\Local\Temp\ipykernel_3908\2983996877.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\hp\AppData\Local\Temp\ipykernel_3908\2983996877.py:15: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer
C:\Users\hp\AppData\Local\Temp\ipykernel_3908\2983996877.py:29: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langcha

Graphe importé avec succés: 


In [2]:
# refresh schema 

graph.refresh_schema()

In [ ]:
# Querying the graph using graph cypher

from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

# GraphCypherQAChain.from_llm(...) : C'est le constructeur de LangChain qui orchestre 
# automatiquement les échanges entre le modèle de langage et la base de données 
# orientée graphe.


# Cette chaine automatise le flux complet en 4 étapes invisibles dès qu' on lui pose
# une question en langage naturel


chain = GraphCypherQAChain.from_llm(
    llm= ChatOllama(model= "llama3.2", temperature= 0),
    graph= graph,
    verbose= True, # affiche dans le terminal tout ce qui se passe en arriére plan 
    allow_dangerous_requests= True
)


# Utilisateur pose une question

# 1. llama3.2 génére la requete Cypher

# 2. exécution de cypher sur Neo4j (via l'objet graph)

# 3. récupération des données brutes

# 4. llama3.2 formule la réponse finale en language naturel

In [7]:
result = chain.invoke({"query": "Quel est le rôle de Guido van Rossum et où travaille-t-il ?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: "Guido van Rossum"})-[:WORKS_AT]->(c:Company)-[:LOCATED_IN]->(l:Location) RETURN p, c, l
Full Context:
[]

> Finished chain.


In [8]:
result

{'query': 'Quel est le rôle de Guido van Rossum et où travaille-t-il ?',
 'result': 'Guido van Rossum a créé Python et travaille à la Mozilla Foundation.'}

In [9]:
result['result']

'Guido van Rossum a créé Python et travaille à la Mozilla Foundation.'

### 2. Improving graph retrieval

#### a.)- Filtering Graph Schema

In [ ]:
llm= ChatOllama(model= "llama3.2", temperature= 0)

chain = GraphCypherQAChain.from_llm(
    llm= llm,
    graph= graph,
    verbose= True,
    allow_dangerous_requests= True,
    exclude_types= ["Concept"]   # FROM EXAMPLE : Exclude nodes with "Concept property "
)

#### b.)- Validating the Cypher Query 

In [ ]:
# when having difficulty in interpreting the direction of relationships:


chain = GraphCypherQAChain.from_llm(
    llm= llm,
    graph= graph,
    verbose= True,
    allow_dangerous_requests= True,
    validate_cypher= True  # !!!!!!!!!!!!!!
)

# validate_cypher: 
# 1. detects nodes and relationships
# 2. Determines the directions of the relationships
# 3. Checks the graph schema
# 4. Update the direction of relationships


In [ ]:
# Another technique to improve the cypher query generation is to use few shot prompting
# by giving few examples like:
examples = ["SOME EXAMPLES"]


from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template(
    "User input: {question}\nCypher query: {query}"
)


cypher_prompt = FewShotPromptTemplate(
    examples= examples,
    example_prompt = example_prompt,
    prefix= "You are a Neo4j expert. Given an input question, create a syntactically correct Cypher query to run.\n\nHere is the schema information\n{schema}.\n\nBelow are a number of examples of questions and their corresponding Cypher queries.",
    suffix= "User input: {question}\nCypher query: ",
    input_variables= ["question"],
)

In [ ]:
# adding few shot examples
chain = GraphCypherQAChain.from_llm(
    graph= graph, 
    llm= llm,
    cypher_prompt= cypher_prompt,
    verbose= True,
    validate_cypher= True,
    allow_dangerous_requests= True
)